### Chi and p-value (least squares modified)

compute my own chi

In [ ]:
### Example linear model

import numpy as np
from scipy.optimize import curve_fit

# Define model
def fit_model(x, m, q):
    return m * x + q

# Get data
file_path = 'data/lens.dat'
x, sigma_x, y, sigma_y = np.loadtxt(file_path, unpack=True)

# Run a first least-square fit ignoring the errors on x.
popt, pcov = curve_fit(fit_model, x, y, sigma=sigma_y)


# Iteratively update the errors and refit  
# to make it more precise, increase the number in 3
for i in range(3):
    
    sigma_eff = np.sqrt(sigma_y**2.0 + (popt[0] * sigma_x)**2.0)
    popt, pcov = curve_fit(fit_model, x, y, sigma=sigma_eff)
    chisq = (((y - fit_model(x, *popt)) / sigma_eff)**2.0).sum()

    # Print the fit output at each step.
    print(f'Step {i}...')
    print(popt, np.sqrt(pcov.diagonal()))
    print(f'Chisquare = {chisq:.2f}')


for any kind of fit

In [ ]:
import numpy as np
import tensorflow as tf
from scipy.optimize import curve_fit

# Define the model (change this for different fits)
def fit_model(x, a, b):
    return a * tf.exp(b * x)  # Example: Exponential function

# Load data
file_path = 'data/lens.dat'
x, sigma_x, y, sigma_y = np.loadtxt(file_path, unpack=True)

# Convert x to TensorFlow tensor
x_tf = tf.convert_to_tensor(x, dtype=tf.float64)

# Compute the first fit (ignoring errors on x)
popt, pcov = curve_fit(fit_model, x, y, sigma=sigma_y)

# Define a function to compute the derivative of the model
def compute_derivative(x_val, params):
    with tf.GradientTape() as tape:
        tape.watch(x_val)
        y_pred = fit_model(x_val, *params)
    return tape.gradient(y_pred, x_val)  # Compute dy/dx

# Iteratively update errors and refit
for i in range(3):
    # Compute the propagated error using automatic differentiation
    df_dx = compute_derivative(x_tf, popt).numpy()  # Convert TF tensor to NumPy array
    sigma_eff = np.sqrt(sigma_y**2 + (df_dx * sigma_x)**2)  # Total effective error
    
    # Refit with updated sigma_eff
    popt, pcov = curve_fit(fit_model, x, y, sigma=sigma_eff)
    
    # Compute chi-square
    chisq = np.sum(((y - fit_model(x, *popt)) / sigma_eff) ** 2)
    
    # Print results
    print(f'Step {i + 1}...')
    print(f'Parameters: {popt}')
    print(f'Errors: {np.sqrt(np.diag(pcov))}')
    print(f'Chi-square: {chisq:.2f}')


complete test

In [5]:
import numpy as np
import tensorflow as tf
from scipy.optimize import curve_fit

# 1. Generate synthetic data (Exponential model with noise)
np.random.seed(42)
x_data = np.linspace(0, 5, 20)  # 20 points from 0 to 5
true_a, true_b = 2.0, 0.8  # True parameters
y_data = true_a * np.exp(true_b * x_data) + np.random.normal(scale=1.0, size=x_data.shape)

# Define uncertainties
sigma_x = 0.1 * np.ones_like(x_data)  # Small uncertainties in x
sigma_y = 0.5 * np.ones_like(y_data)  # Measurement noise

# Convert x_data to a TensorFlow tensor
x_tf = tf.convert_to_tensor(x_data, dtype=tf.float64)


# 2. Define the fit model (Exponential function)
def fit_model(x, a, b):
    return a * tf.exp(b * x)  # Exponential function


# 3. Compute the first fit (ignoring errors on x)
popt, pcov = curve_fit(fit_model, x_data, y_data, sigma=sigma_y)


# 4. Define function to compute the derivative of the model
def compute_derivative(x_val, params):
    with tf.GradientTape() as tape:
        tape.watch(x_val)
        y_pred = fit_model(x_val, *params)
    return tape.gradient(y_pred, x_val)  # Compute dy/dx

# 5. Iteratively update errors and refit
for i in range(3):
    # Compute propagated error using automatic differentiation
    df_dx = compute_derivative(x_tf, popt).numpy()  # Convert TF tensor to NumPy array
    sigma_eff = np.sqrt(sigma_y**2 + (df_dx * sigma_x)**2)  # Total effective error
    
    # Refit with updated sigma_eff
    popt, pcov = curve_fit(fit_model, x_data, y_data, sigma=sigma_eff)
    
    # Compute chi-square
    chisq = np.sum(((y_data - fit_model(x_data, *popt)) / sigma_eff) ** 2)
    
    # Print results
    print(f'Step {i + 1}...')
    print(f'Parameters: {popt}')
    print(f'Errors: {np.sqrt(np.diag(pcov))}')
    print(f'Chi-square: {chisq:.2f}')


2025-03-11 09:16:21.000237: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-11 09:16:21.022149: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-11 09:16:21.214506: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-11 09:16:21.428868: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-11 09:16:21.618703: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been 

Step 1...
Parameters: [2.22969125 0.76809782]
Errors: [0.10285692 0.01391644]
Chi-square: 12.65
Step 2...
Parameters: [2.22203607 0.76934626]
Errors: [0.1018184  0.01352152]
Chi-square: 12.49
Step 3...
Parameters: [2.22246029 0.7692824 ]
Errors: [0.10187025 0.01353966]
Chi-square: 12.48


after computing chi, do test

In [1]:
import scipy.stats

dof = 12
value = 5.0

chi2 = scipy.stats.chi2(dof)
p_value = chi2.cdf(value)

print(p_value)
print(chi2.ppf(p_value))

0.04202103819530614
5.0


### Gaussian distribution

$N(z; \mu, \sigma)$  

In [2]:
import scipy.stats

mu = 0.0
sigma = 1.0
z = scipy.stats.norm(mu, sigma)

# Probability density function (pdf), compute prob that exactly p(z=0.0)
pdf_value = z.pdf(0.0)

# Cumulative distribution function (CDF), compute p(z ≤ 0)
cdf_value = z.cdf(0)

# Generate 10 samples
samples = z.rvs(size=10)

0.3989422804014327
1.0 -> 0.6826894921370859
2.0 -> 0.9544997361036416
3.0 -> 0.9973002039367398


In [3]:
# Compute prob that a sample z lies within 1,2,3 standard deviation from the mean
# it's the 60, 95, 99 rule  

# P = z.cdf(z0) - z.cdf(-z0) 
# means prob(z<z0)-prob(z<-z0) = prob(-z0 <z<z0)

for z0 in [1.0, 2.0, 3.0]:
    P = z.cdf(z0) - z.cdf(-z0)
    print(f'{z0} -> {P}')

1.0 -> 0.6826894921370859
2.0 -> 0.9544997361036416
3.0 -> 0.9973002039367398


In [ ]:
# self-made: cumulative function z.cdf(z0)

import numpy as np

def Phi(z):
    """ Gaussian cumulative function.
    """
    return 0.5 + 0.5 * np.math.erf(z / np.sqrt(2.0))

def integrate_gauss(x1, x2, mu=0.0, sigma=1.0):
    """Integrate a generic gaussian between x1 and x2.
    """
    z1 = (x1 - mu) / sigma
    z2 = (x2 - mu) / sigma
    return Phi(z2) - Phi(z1)

print(integrate_gauss(-1.0, 1.0))
print(integrate_gauss(22.0, 24.0, 20.0, 4.0))


### Poisson

In [ ]:
import numpy as np
import scipy.stats

mu = 2.
k = np.linspace(0, 3, 4, dtype=int)
print(k)
print(scipy.stats.poisson(mu).pmf(k))

### Weighted average

In [ ]:
x = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
w = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

m = np.average(x, weights=w)

In [ ]:
# self-made

import numpy as np

def weighted_average(y, sigma_y):
    """Implementation of the weighted average using numpy.
    """
    w = 1.0 / sigma_y**2.0
    q_hat = np.average(y, weights=w)
    sigma_q = np.sqrt(1.0 / np.sum(w))
    return q_hat, sigma_q

n = np.array([1.325, 1.36, 1.32, 1.338, 1.335])
sigma_n = np.array([0.012, 0.05, 0.01, 0.005, 0.006])
q_hat, sigma_q = weighted_average(n, sigma_n)
print(f'q = {q_hat:.4f} +/- {sigma_q:.4f}')

### Binomial

In [ ]:
import numpy as np
import scipy.stats

n = 4
p = 0.1
k = np.linspace(0, n, n + 1, dtype=int)
print(k)
print(scipy.stats.binom(n, p).pmf(k))

In [ ]:
# self-made

from math import factorial

def binomial(k, n, p):
    binom = factorial(n) / (factorial(k) * factorial(n - k))
    return binom * (p**k)*((1.0 - p)**(n - k))

for k in range(4):
    print(binomial(k, 4, 1.0 / 6.0))

